# Patrons de conception

## Observateur

Une boutique veut déclencher plusieurs actions quand une commande est payée (envoyer un reçu, mettre à jour le stock…), sans que le code de paiement connaisse ces actions.

Écrivez une classe `EventBus` avec deux méthodes&nbsp;:

- `subscribe(event, callback)`, qui abonne une fonction à un événement&nbsp;;
- `publish(event, *args)`, qui appelle avec `args`, dans l'ordre d'abonnement, toutes les fonctions abonnées à cet événement.

Abonnez `send_receipt` et `update_stock` à l'événement `"order_paid"`, puis publiez-le pour la commande 42&nbsp;: les deux messages doivent s'afficher. Publier un événement sans abonné ne doit rien faire.

In [ ]:
def send_receipt(order_id: int) -> None:
    print(f"Reçu envoyé pour la commande {order_id}")


def update_stock(order_id: int) -> None:
    print(f"Stock mis à jour pour la commande {order_id}")

In [ ]:
# Votre code ici

### Solution

In [ ]:
from collections import defaultdict
from collections.abc import Callable


class EventBus:
    def __init__(self) -> None:
        self._subscribers: defaultdict[str, list[Callable[..., None]]] = defaultdict(
            list
        )

    def subscribe(self, event: str, callback: Callable[..., None]) -> None:
        self._subscribers[event].append(callback)

    def publish(self, event: str, *args: object) -> None:
        for callback in self._subscribers.get(event, []):
            callback(*args)


bus = EventBus()
bus.subscribe("order_paid", send_receipt)
bus.subscribe("order_paid", update_stock)
bus.publish("order_paid", 42)
bus.publish("order_cancelled", 42)  # Aucun abonné : rien ne se passe

Le code qui publie l'événement ne dépend d'aucun abonné&nbsp;: on ajoute une action en abonnant une fonction, sans modifier le code de paiement.

## Itérateur

Implémentez le patron itérateur pour la classe `Tree` avec le parcours en profondeur (DFS).

In [ ]:
from dataclasses import dataclass
from typing import Self


@dataclass
class Tree[T]:
    value: T
    children: list[Self]


tree = Tree("a", [Tree("b", [Tree("c", [])]), Tree("d", [])])
# print(f"DFS : {list(tree)}")

### Solution

In [ ]:
from collections.abc import Iterator
from dataclasses import dataclass
from typing import Self


@dataclass
class Tree[T]:
    value: T
    children: list[Self]

    def __iter__(self) -> Iterator[T]:
        yield self.value
        for child in self.children:
            yield from iter(child)


# Option avec classe externe
class DepthFirstSearch[T](Iterator[T]):
    def __init__(self, tree: Tree[T]) -> None:
        self._iterator = self._iterate(tree)

    @classmethod
    def _iterate(cls, tree: Tree[T]) -> Iterator[T]:
        yield tree.value
        for child in tree.children:
            yield from cls._iterate(child)

    def __next__(self) -> T:
        return next(self._iterator)


@dataclass
class Tree[T]:
    value: T
    children: list[Self]

    def __iter__(self) -> Iterator[T]:
        return DepthFirstSearch(self)


tree = Tree("a", [Tree("b", [Tree("c", [])]), Tree("d", [])])
print(f"DFS : {list(tree)}")